In [1]:
from docx import Document
from docx.enum.style import WD_STYLE_TYPE
from docx.oxml.ns import qn
from docx.oxml.parser import OxmlElement
from docx.shared import Pt


def create_hyperlink(paragraph, text, anchor):
    """
    Create a hyperlink within a paragraph to a bookmark
    """
    # Create the w:hyperlink element
    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("w:anchor"), anchor)
    # Create a new run
    new_run = OxmlElement("w:r")
    # Create a new text element
    rPr = OxmlElement("w:rPr")
    new_run.append(rPr)
    t = OxmlElement("w:t")
    t._setText(text)
    new_run.append(t)
    hyperlink.append(new_run)
    # Append the hyperlink to the paragraph
    paragraph._p.append(hyperlink)
    return hyperlink


def create_document_from_qna(qna):
    doc = Document()

    # Add title
    title = doc.add_heading("Q&A Document", level=0)
    title.alignment = 1  # Center alignment

    # Create styles
    styles = doc.styles
    question_style = styles.add_style("Question", WD_STYLE_TYPE.PARAGRAPH)
    question_style.font.bold = True
    question_style.font.size = Pt(14)
    answer_style = styles.add_style("Answer", WD_STYLE_TYPE.PARAGRAPH)
    answer_style.font.size = Pt(12)
    source_style = styles.add_style("Source", WD_STYLE_TYPE.PARAGRAPH)
    source_style.font.size = Pt(10)
    source_style.font.italic = True

    # Add questions and answers
    for i, (question, answer, sources) in enumerate(
        zip(qna["questions"], qna["answers"], qna["sources"]), 1
    ):
        doc.add_paragraph(f"Question {i}: {question}", style="Question")
        answer_para = doc.add_paragraph(answer + "\n", style="Answer")
        # Add source references
        for j, source in enumerate(sources, 1):
            create_hyperlink(answer_para, f"[{j}]", f"source_{i}_{j}")
            if j < len(sources):
                answer_para.add_run(", ")
        doc.add_paragraph()  # Add a blank line

    # Add sources section
    doc.add_page_break()
    sources_heading = doc.add_heading("Sources", level=1)
    sources_heading.runs[0].underline = True

    for i, question_sources in enumerate(qna["sources"], 1):
        # Add subheading for each question's sources
        doc.add_heading(f"Question {i}", level=2)

        for j, source in enumerate(question_sources, 1):
            bookmark_name = f"source_{i}_{j}"
            source_para = doc.add_paragraph(style="Source")
            source_run = source_para.add_run(f"{source['name']}")
            source_run.bold = True
            # Add bookmark
            bookmark_start = OxmlElement("w:bookmarkStart")
            bookmark_start.set(qn("w:id"), f"{i}{j}")
            bookmark_start.set(qn("w:name"), bookmark_name)
            source_para._p.insert(0, bookmark_start)
            bookmark_end = OxmlElement("w:bookmarkEnd")
            bookmark_end.set(qn("w:id"), f"{i}{j}")
            bookmark_end.set(qn("w:name"), bookmark_name)
            source_para._p.append(bookmark_end)
            doc.add_paragraph(source["text"], style="Source")
            # doc.add_paragraph()  # Add a blank line

    doc.save("qna_document.docx")


In [2]:
# Example usage
# Sample QnA data
qna = {
    "questions": ["Who won the 2022 FIFA World Cup?", "What is the capital of China?"],
    "answers": [
        "- Argentina won the 2022 FIFA World Cup.\n- The final was against France.",
        "- The capital of China is Beijing.\n- Estimated population: 21.54 million.",
    ],
    "sources": [
        [
            {
                "name": "FIFA Official Website",
                "text": "Argentina won the 2022 FIFA World Cup after a thrilling final match against France.",
            },
            {
                "name": "Sports News",
                "text": "Lionel Messi led Argentina to victory in the 2022 World Cup, defeating France in penalties.",
            },
        ],
        [
            {
                "name": "Geography Encyclopedia",
                "text": "Beijing is the capital of China, with an estimated population of 21.54 million as of 2021.",
            }
        ],
    ],
}

create_document_from_qna(qna)

AttributeError: 'CT_Text' object has no attribute '_setText'